In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("/content/drive/MyDrive/clean_retention_features_updated_new.csv")
print(df.shape)
df.head()

(1676, 20)


,num__DistanceFromHome,num__JobSatisfaction,num__EnvironmentSatisfaction,num__WorkLifeBalance,num__RelationshipSatisfaction,num__JobInvolvement,num__TrainingTimesLastYear,num__YearsSinceLastPromotion,num__YearsInCurrentRole,num__TotalWorkingYears,num__JobLevel,num__MonthlyIncome,num__PercentSalaryHike,num__PerformanceRating,num__YearsWithCurrManager,cat__Department_Cardiology,cat__Department_Maternity,cat__Department_Neurology,cat__JobRoleGroup_Clinical,OverTime
0,0.000000,1.000000,0.333333,0.000000,0.000000,0.666667,0.083333,0.000000,0.222222,0.200,0.25,0.262454,0.000000,0.0,0.294118,1.0,0.0,0.0,1.0,0.0
1,0.250000,0.333333,0.666667,0.666667,1.000000,0.333333,0.500000,0.066667,0.388889,0.250,0.25,0.217009,0.857143,0.0,0.411765,0.0,1.0,0.0,0.0,1.0
2,0.035714,0.666667,1.000000,0.666667,0.333333,0.333333,0.500000,0.000000,0.000000,0.175,0.00,0.056925,0.285714,0.0,0.000000,0.0,1.0,0.0,1.0,0.0
3,0.071429,0.666667,1.000000,0.666667,0.666667,0.666667,0.500000,0.200000,0.388889,0.200,0.00,0.100053,0.000000,0.0,0.000000,0.0,1.0,0.0,0.0,1.0
4,0.035714,0.333333,0.000000,0.666667,1.000000,0.666667,0.500000,0.133333,0.111111,0.150,0.00,0.129489,0.071429,0.0,0.117647,0.0,1.0,0.0,1.0,0.0


In [3]:
FEATURES_FOR_STRATEGY = [
    "num__JobSatisfaction",
    "num__WorkLifeBalance",
    "num__MonthlyIncome",
    "num__JobInvolvement",
    "num__TrainingTimesLastYear",
    "num__DistanceFromHome",
    "num__EnvironmentSatisfaction",
    "num__RelationshipSatisfaction",
    "num__PerformanceRating",
    "num__YearsSinceLastPromotion",
    "num__YearsInCurrentRole",
    "num__TotalWorkingYears",
    "num__JobLevel",
    "OverTime"
]

In [4]:
# 3️ BUILD QUANTILE-BASED CLUSTERING MODEL (BALANCED)

def build_feature_cluster_quantile(df, feature):

    # Splits feature into balanced Low / Medium / High groups using quantiles to avoid clustering bias.

    values = pd.to_numeric(df[feature], errors="coerce")

    q1 = values.quantile(0.33)
    q2 = values.quantile(0.66)

    return {
        "q1": q1,
        "q2": q2
    }

# Train quantile thresholds for each feature
feature_models = {}
for feature in FEATURES_FOR_STRATEGY:
    feature_models[feature] = build_feature_cluster_quantile(df, feature)

In [5]:
# RETENTION STRATEGY MAP

LEVEL_STRATEGY_MAP = {
    "num__JobSatisfaction": {
        "Low": ["Urgent engagement intervention"],
        "Medium": ["Career discussions"],
        "High": ["Recognition programs"]
    },
    "num__WorkLifeBalance": {
        "Low": ["Policy-level work-life redesign"],
        "Medium": ["Workload reassessment"],
        "High": ["Flexible scheduling"]
    },
    "num__MonthlyIncome": {
        "Low": ["Salary revision"],
        "Medium": ["Compensation benchmarking"],
        "High": ["Performance incentives"]
    },
    "num__JobInvolvement": {
        "Low": ["Job redesign and mentoring"],
        "Medium": ["Team engagement activities"],
        "High": ["Leadership opportunities"]
    },
    "num__TrainingTimesLastYear": {
        "Low": ["Immediate upskilling initiatives"],
        "Medium": ["Skill development programs"],
        "High": ["Advanced professional training"]
    },
    "num__DistanceFromHome": {
        "Low": ["Remote work or relocation support"],
        "Medium": ["Commute flexibility"],
        "High": ["Hybrid work options"]
    },
    "num__EnvironmentSatisfaction": {
        "Low": ["Immediate workplace condition review"],
        "Medium": ["Environment feedback sessions"],
        "High": ["Workplace improvements"]
    },
    "num__RelationshipSatisfaction": {
        "Low": ["Conflict resolution support"],
        "Medium": ["Manager mediation"],
        "High": ["Team bonding initiatives"]
    },
    "num__PerformanceRating": {
        "Low": ["Skill gap assessment and mentoring"],
        "Medium": ["Performance coaching"],
        "High": ["High-performance incentives"]
    },
    "num__YearsSinceLastPromotion": {
        "Low": ["Immediate promotion review"],
        "Medium": ["Career roadmap discussion"],
        "High": ["Fast-track promotion consideration"]
    },
    "num__YearsInCurrentRole": {
        "Low": ["Role change opportunity"],
        "Medium": ["Job rotation"],
        "High": ["Role enrichment"]
    },
    "num__TotalWorkingYears": {
        "Low": ["Long-term career planning"],
        "Medium": ["Career stability incentives"],
        "High": ["Retention bonuses"]
    },
    "num__JobLevel": {
        "Low": ["Level-up evaluation"],
        "Medium": ["Role responsibility expansion"],
        "High": ["Leadership development"]
    },
    "OverTime": {
        "Low": ["Overtime reduction and staffing support"],
        "Medium": ["Shift balancing"],
        "High": ["Maintain current workload"]
    }
}


In [6]:
#  ASSIGN LEVEL FOR A SINGLE FEATURE VALUE

def get_feature_level(feature, value):
    model = feature_models[feature]
    q1, q2 = model["q1"], model["q2"]

    if value <= q1:
        return "Low"
    elif value <= q2:
        return "Medium"
    else:
        return "High"


In [7]:
# RECOMMEND STRATEGIES FOR ONE FEATURE

def recommend_strategy(feature_name, feature_value):
    level = get_feature_level(feature_name, feature_value)

    return {
        "Feature": feature_name,
        "Cluster_Level": level,
        "Recommended_Strategies": LEVEL_STRATEGY_MAP[feature_name][level]
    }

In [8]:
# RECOMMEND STRATEGIES FOR AN EMPLOYEE (MULTIPLE FEATURES)

def recommend_strategies_for_employee(employee_features):
    strategies = set()
    cluster_summary = {}

    for feature, value in employee_features.items():
        result = recommend_strategy(feature, value)
        cluster_summary[feature] = result["Cluster_Level"]
        strategies.update(result["Recommended_Strategies"])

    return {
        "Cluster_Assignments": cluster_summary,
        "Final_Retention_Strategies": list(strategies)
    }

In [9]:
# MULTI-EMPLOYEE SUPPORT (WITH ID)

def recommend_strategies_for_multiple_employees(employee_df, id_column="EmployeeID"):
    results = {}

    for _, row in employee_df.iterrows():
        employee_id = row[id_column]
        employee_features = row.drop(id_column).to_dict()

        results[employee_id] = recommend_strategies_for_employee(employee_features)

    return results

In [10]:
# EXAMPLE INPUT (MULTIPLE EMPLOYEES)

employee_data = pd.DataFrame([
    {
        "EmployeeID": 101,
        "num__RelationshipSatisfaction": 0.4,
        "num__JobSatisfaction": 0.1
    },
    {
        "EmployeeID": 102,
        "num__RelationshipSatisfaction": 0.8,
        "num__JobSatisfaction": 0.6
    }
])


In [11]:
employee_results = recommend_strategies_for_multiple_employees(employee_data)
employee_results

{np.float64(101.0): {'Cluster_Assignments': {'num__RelationshipSatisfaction': 'Medium',
   'num__JobSatisfaction': 'Low'},
  'Final_Retention_Strategies': ['Manager mediation',
   'Urgent engagement intervention']},
 np.float64(102.0): {'Cluster_Assignments': {'num__RelationshipSatisfaction': 'High',
   'num__JobSatisfaction': 'Medium'},
  'Final_Retention_Strategies': ['Team bonding initiatives',
   'Career discussions']}}